# broadcast-source-fanout — worked example 2: Fan a VQ codebook out to a sequence of code indices

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `broadcast-source-fanout`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

Vector-quantized models store a `(K, D)` codebook of K learned vectors. A sequence of integer codes is decoded by indexing the codebook: `codebook[codes]` fans each code id out to its full D-dimensional vector. Repeated code ids select the same row — the broadcast/fan-out semantics. This is the same one-source-to-many copy pattern as a distributed broadcast, expressed as advanced integer indexing.

## Worked solution

**Goal.** Given a `(K, D)` codebook and a `(B, L)` grid of integer code ids, reconstruct the `(B, L, D)` tensor of decoded vectors.

1. **Understand the source.** The codebook is the single source of truth: K distinct D-vectors. Every output element is a *copy* of one codebook row — nothing is computed, only fanned out.
2. **Advanced integer indexing.** `codebook[codes]` where `codes` has shape `(B, L)` produces shape `(B, L, D)`: PyTorch replaces each integer in `codes` with the corresponding length-D row. The leading index dimensions `(B, L)` are preserved and the indexed dimension's size `D` is appended.
3. **Why repeated ids share a row.** Two positions holding the same code id both index the same physical codebook row, so they receive identical D-vectors — this is exactly the fan-out (one source row → many destinations) the atom is about.
4. **Dtype preservation.** Indexing copies values, so the output dtype matches the codebook dtype (float32 here), and gradients would flow back to the indexed rows only.
5. **Verification.** We rebuild the answer with an explicit Python loop over every `(b, l)` position and assert it matches the vectorized index, confirming the fan-out is correct.

In [ ]:
import torch as t
from torch import Tensor

def decode_codes(codebook: Tensor, codes: Tensor) -> Tensor:
    # codebook: (K, D), codes: (B, L) int64 -> (B, L, D)
    return codebook[codes]

t.manual_seed(0)
K, D, B, L = 5, 4, 2, 3
codebook = t.randn(K, D)
codes = t.randint(0, K, (B, L))
out = decode_codes(codebook, codes)
print("out shape:", tuple(out.shape))
# explicit reference
ref = t.stack([t.stack([codebook[codes[b, l]] for l in range(L)]) for b in range(B)])
print("matches loop reference:", bool(t.allclose(out, ref)))